# Chấm điểm động tác chống đẩy — Bước 4: baseline LSTM

Đầu vào: `pushup_dataset.npz` của bước 3.

Mô hình: LSTM hai chiều đọc chuỗi 32 khung, mỗi khung là vector 12 khớp × (x, y, visibility), phân loại rep **đúng (0) / sai (1)**.

Đánh giá bằng 5-fold chia theo người tập: fold `k` làm test, fold `k+1` làm validation để chọn epoch, 3 fold còn lại để train.
Mọi rep đều được dự đoán đúng một lần khi nó nằm ở fold test, nên các chỉ số bên dưới tính trên toàn bộ dữ liệu
mà mô hình chưa từng thấy người đó.

## 1. Đọc dữ liệu

In [ ]:
import os, glob, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

found = glob.glob("/kaggle/working/**/pushup_dataset.npz", recursive=True) \
      + glob.glob("/kaggle/input/**/pushup_dataset.npz", recursive=True)
assert found, "Không thấy pushup_dataset.npz. Hãy Add Input dataset của bước 3."
DATA_PATH = found[0]
OUT_DIR = "/kaggle/working/lstm_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

data = np.load(DATA_PATH)
X = data["X2d"]                      # (N, T, 12, 3): x, y, visibility
y, view, subject, fold = data["y"], data["view"], data["subject"], data["fold"]
VIEW_NAMES = ["front", "side", "diagonal"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("X:", X.shape, "| thiết bị:", device)
print("Tỉ lệ rep sai:", round(float(y.mean()), 3), "→ đoán bừa 'đúng' hết sẽ được accuracy", round(float(1 - y.mean()), 3))

## 2. Tăng cường dữ liệu (chỉ áp dụng khi train)

Dữ liệu chỉ có 24 người nên mô hình rất dễ học thuộc từng người. Mỗi lần lấy một rep để train, rep đó được biến đổi ngẫu nhiên:
- **Lật trái/phải**: đổi dấu x và hoán đổi khớp trái ↔ phải (camera đặt bên trái hay bên phải đều như nhau).
- **Co giãn và xoay nhẹ**: ±10% kích thước, ±10° (camera đặt hơi lệch).
- **Joint dropout**: che ngẫu nhiên một số khớp (toạ độ và visibility về 0) để mô hình chịu được khớp bị che.

In [ ]:
SWAP_LR = [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10]      # chỉ số khớp sau khi hoán đổi trái ↔ phải


def augment(x, rng):
    x = x.copy()                                       # (T, V, 3)
    if rng.random() < 0.5:
        x = x[:, SWAP_LR]
        x[..., 0] = -x[..., 0]
    scale = rng.uniform(0.9, 1.1)
    a = np.deg2rad(rng.uniform(-10, 10))
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]], dtype=np.float32)
    x[..., :2] = (x[..., :2] @ rot.T) * scale
    drop = rng.random(x.shape[1]) < 0.1                # mỗi khớp có 10% khả năng bị che trong cả rep
    x[:, drop] = 0.0
    return x


class RepDataset(torch.utils.data.Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = idx, train
        self.rng = np.random.default_rng(0)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = augment(X[j], self.rng) if self.train else X[j]
        return torch.from_numpy(x), int(view[j]), int(y[j])

## 3. Mô hình

Mỗi khung hình: 12 khớp × 3 = 36 số, ghép thêm 3 số one-hot của góc quay (để một mô hình dùng chung cho cả 3 góc).
LSTM hai chiều 2 lớp → lấy trung bình theo thời gian → lớp tuyến tính ra 1 số (logit của lớp "sai").

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_joints=12, n_channels=3, hidden=64, n_views=3, dropout=0.3):
        super().__init__()
        self.n_views = n_views
        self.lstm = nn.LSTM(n_joints * n_channels + n_views, hidden, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden * 2, 1))

    def forward(self, x, view_id):
        b, t = x.shape[:2]
        v = nn.functional.one_hot(view_id, self.n_views).float()[:, None].expand(b, t, self.n_views)
        out, _ = self.lstm(torch.cat([x.reshape(b, t, -1), v], dim=-1))
        return self.head(out.mean(dim=1)).squeeze(-1)


print("Số tham số:", sum(p.numel() for p in LSTMClassifier().parameters()))

## 4. Hàm train và đánh giá

- Hàm mất mát: binary cross-entropy. Tối ưu: AdamW.
- Sau mỗi epoch đo loss trên validation; giữ lại trọng số tốt nhất; dừng sớm nếu 12 epoch liền không cải thiện.
- `views`: danh sách góc quay được dùng. `[0, 1, 2]` = một mô hình chung; `[1]` = mô hình riêng cho góc ngang.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


@torch.no_grad()
def predict(model, idx):
    model.eval()
    loader = torch.utils.data.DataLoader(RepDataset(idx, train=False), batch_size=256)
    probs = [torch.sigmoid(model(xb.to(device), vb.to(device))).cpu() for xb, vb, _ in loader]
    return torch.cat(probs).numpy()


def train_one_fold(make_model, k, views, epochs=80, patience=12, lr=1e-3, seed=0):
    set_seed(seed + k)
    use = np.isin(view, views)
    test_idx = np.where(use & (fold == k))[0]
    val_idx = np.where(use & (fold == (k + 1) % 5))[0]
    train_idx = np.where(use & (fold != k) & (fold != (k + 1) % 5))[0]

    model = make_model().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = torch.utils.data.DataLoader(RepDataset(train_idx, train=True), batch_size=64, shuffle=True)
    y_val = torch.tensor(y[val_idx], dtype=torch.float32)

    best_loss, best_state, bad_epochs, history = np.inf, None, 0, []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, vb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb.to(device), vb.to(device)), yb.float().to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(yb)
        p_val = torch.tensor(predict(model, val_idx))
        val_loss = nn.functional.binary_cross_entropy(p_val, y_val).item()
        history.append((total / len(train_idx), val_loss))
        if val_loss < best_loss:
            best_loss, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, test_idx, predict(model, test_idx), history


def run_cv(make_model, views, name):
    prob = np.full(len(y), np.nan, dtype=np.float32)
    histories = []
    for k in range(5):
        model, test_idx, p, history = train_one_fold(make_model, k, views)
        prob[test_idx] = p
        histories.append(history)
        acc = ((p > 0.5) == y[test_idx]).mean()
        print(f"[{name}] fold {k}: {len(history)} epoch, accuracy test = {acc:.3f}")
        torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{name}_fold{k}.pt"))
    return prob, histories


def metrics(prob, mask, threshold=0.5):
    # Lớp dương = rep SAI. false_alarm = tỉ lệ rep ĐÚNG bị báo là sai.
    t, p = y[mask], (prob[mask] > threshold).astype(int)
    tp, fp = int(((p == 1) & (t == 1)).sum()), int(((p == 1) & (t == 0)).sum())
    fn, tn = int(((p == 0) & (t == 1)).sum()), int(((p == 0) & (t == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return dict(n=int(mask.sum()), accuracy=(tp + tn) / max(len(t), 1), precision=precision, recall=recall,
                f1=2 * precision * recall / max(precision + recall, 1e-9), false_alarm=fp / max(fp + tn, 1),
                TP=tp, FP=fp, FN=fn, TN=tn)


def report(prob, name):
    rows = {f"{name} / tất cả": metrics(prob, ~np.isnan(prob))}
    for v, view_name in enumerate(VIEW_NAMES):
        mask = ~np.isnan(prob) & (view == v)
        if mask.any():
            rows[f"{name} / {view_name}"] = metrics(prob, mask)
    return pd.DataFrame(rows).T

## 5. Phương án B: một mô hình chung cho cả 3 góc quay

In [ ]:
prob_common, hist_common = run_cv(LSTMClassifier, views=[0, 1, 2], name="lstm_chung")
table_common = report(prob_common, "LSTM chung")
display(table_common.round(3))

## 6. Phương án A: mỗi góc quay một mô hình riêng

In [ ]:
prob_separate = np.full(len(y), np.nan, dtype=np.float32)
for v, view_name in enumerate(VIEW_NAMES):
    p, _ = run_cv(LSTMClassifier, views=[v], name=f"lstm_{view_name}")
    prob_separate[view == v] = p[view == v]
table_separate = report(prob_separate, "LSTM riêng")
display(table_separate.round(3))

## 7. So sánh và lưu kết quả

In [ ]:
result = pd.concat([table_common, table_separate])
result.to_csv(os.path.join(OUT_DIR, "lstm_results.csv"))
np.savez(os.path.join(OUT_DIR, "lstm_predictions.npz"), prob_common=prob_common, prob_separate=prob_separate, y=y,
         view=view, subject=subject, fold=fold)
display(result[["n", "accuracy", "precision", "recall", "f1", "false_alarm"]].round(3))

## 8. Đường cong học và kết quả theo từng người

- Đường cong học: nếu loss train giảm mãi mà loss validation tăng lên thì mô hình đang học thuộc (overfit).
- Accuracy theo từng người: cho biết mô hình sai đều hay chỉ sai nặng ở vài người (có thể do nhãn hoặc do video của người đó).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(22, 3.5), sharey=True)
for k, history in enumerate(hist_common):
    history = np.array(history)
    axes[k].plot(history[:, 0], label="train")
    axes[k].plot(history[:, 1], label="validation")
    axes[k].set_title(f"fold {k}"); axes[k].set_xlabel("epoch"); axes[k].grid(alpha=0.3)
axes[0].set_ylabel("loss"); axes[0].legend()
plt.tight_layout(); plt.show()

per_subject = pd.DataFrame({"subject": subject, "dung": (prob_common > 0.5) == y, "view": np.array(VIEW_NAMES)[view]})
display(per_subject.pivot_table(index="view", columns="subject", values="dung", aggfunc="mean").round(2))

## 9. Kết quả

Tải thư mục `lstm_baseline` (bảng kết quả, dự đoán, trọng số) hoặc Save Version để giữ lại.

In [ ]:
!ls -lh "{OUT_DIR}"